# 02 — Data Preprocessing
## Làm Sạch và Chuẩn Bị Dữ Liệu

**Mục tiêu**: Chuyển dữ liệu thô (`train.csv`, `test.csv`) thành một bộ dữ liệu sạch, đã mã hóa và nhất quán — sẵn sàng cho giai đoạn **Feature Engineering (NB03)**.

**Pipeline tổng quan**:
1. Load dữ liệu & lưu ID
2. Drop cột không liên quan (`id`, `Name`)
3. Làm sạch giá trị bẩn (`Sleep Duration`, `Dietary Habits`)
4. Xử lý missing values thưa
5. Mã hóa binary features
6. Merge conditional columns → `Pressure`, `Satisfaction`
7. Mã hóa ordinal features
8. Validation & lưu kết quả

---
## 2.1 — EDA Recap: Các vấn đề cần xử lý

Tóm tắt các phát hiện từ `01_eda.ipynb` có liên quan trực tiếp đến preprocessing:

| # | Vấn đề | Chi tiết | Hướng xử lý |
|---|--------|----------|-------------|
| 1 | **Giá trị bẩn — `Sleep Duration`** | 36 unique thay vì 4. ~79 mẫu chứa tên thành phố, tên cột (`Pune`, `Work_Study_Hours`, v.v.) | Map về 4 category hợp lệ, còn lại → mode |
| 2 | **Giá trị bẩn — `Dietary Habits`** | 23 unique thay vì 3. ~27 mẫu chứa tên bằng cấp, giới tính (`BSc`, `Male`, v.v.) | Map về 3 category hợp lệ, còn lại → mode |
| 3 | **Conditional missing (Student)** | `Academic Pressure`, `CGPA`, `Study Satisfaction` missing ~100% ở Working Professional | Merge thành `Pressure`, `Satisfaction`; `has_cgpa` flag |
| 4 | **Conditional missing (Professional)** | `Work Pressure`, `Job Satisfaction` missing ~100% ở Student | Cùng merge ở bước 3 |
| 5 | **Missing thưa** | `Dietary Habits` (4), `Financial Stress` (4), `Degree` (2) | Fill median / mode / constant |
| 6 | **ID columns** | `id`, `Name` không có ý nghĩa dự đoán | Drop (lưu `id` riêng cho submission) |
| 7 | **High cardinality** | `City` (98), `Profession` (64), `Degree` (115) | Giữ nguyên string → Target Encoding ở NB03 |
| 8 | **Class imbalance** | 81.83% vs 18.17% | Không xử lý ở đây — dùng `class_weight='balanced'` ở modeling |
| 9 | **Scaling** | LR và SVM cần scale; tree-based thì không | Không scale ở đây — inject qua `Pipeline` ở NB04 |

---
## 2.2 — Setup & Load Data

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Add project root to path so src/ is importable
sys.path.insert(0, os.path.abspath('..'))
from src.preprocessing_utils import (
    extract_ids,
    drop_irrelevant_columns,
    clean_categorical_column,
    fill_sparse_missing,
    encode_binary_features,
    merge_conditional_columns,
    encode_ordinal_features,
    validate_preprocessed_data,
    save_processed_data,
)

RANDOM_STATE = 42

# Load raw data
train_raw = pd.read_csv('../data/raw/train.csv')
test_raw  = pd.read_csv('../data/raw/test.csv')

print(f'Train: {train_raw.shape[0]:,} rows × {train_raw.shape[1]} cols')
print(f'Test:  {test_raw.shape[0]:,} rows × {test_raw.shape[1]} cols')
train_raw.head(3)

Train: 140,700 rows × 20 cols
Test:  93,800 rows × 19 cols


,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,0,Aaradhya,Female,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No,0
1,1,Vivan,Male,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No,1
2,2,Yuvraj,Male,33.0,Visakhapatnam,Student,NaN,5.0,NaN,8.97,2.0,NaN,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1


---
## 2.3 — Drop Irrelevant Columns & Preserve IDs

- **`id`**: Định danh kỹ thuật, không có ý nghĩa dự đoán. Lưu lại riêng để dùng khi tạo file submission ở NB06.
- **`Name`**: Tên cá nhân — không mang thông tin có thể tổng quát hóa cho model.

In [2]:
# Preserve IDs before dropping
train_ids = extract_ids(train_raw, id_col='id')
test_ids  = extract_ids(test_raw,  id_col='id')

COLS_TO_DROP = ['id', 'Name']

print('Train:')
train = drop_irrelevant_columns(train_raw, COLS_TO_DROP)
print('Test:')
test  = drop_irrelevant_columns(test_raw,  COLS_TO_DROP)

print(f'\nTrain shape after drop: {train.shape}')
print(f'Test  shape after drop: {test.shape}')

Train:
  Dropped 2 column(s): ['id', 'Name']
Test:
  Dropped 2 column(s): ['id', 'Name']

Train shape after drop: (140700, 18)
Test  shape after drop: (93800, 17)


---
## 2.4 — Làm Sạch Giá Trị Bẩn

EDA phát hiện 2 cột có dữ liệu bị nhiễu do lỗi nhập liệu (data từ cột khác bị swap vào):

### `Sleep Duration`
4 category hợp lệ (dựa trên bivariate analysis — có gradient rõ ràng với Depression):

| Category | Depression rate |
|----------|-----------------|
| Less than 5 hours | 23.5% |
| 5-6 hours | 16.6% |
| 7-8 hours | 17.8% |
| More than 8 hours | 13.9% |

### `Dietary Habits`
3 category hợp lệ (gradient rõ ràng với Depression):

| Category | Depression rate |
|----------|-----------------|
| Unhealthy | 26.1% |
| Moderate | 16.6% |
| Healthy | 11.8% |

**Chiến lược**: Bất kỳ giá trị nào không thuộc tập hợp lệ → `NaN` → fill bằng mode của các giá trị hợp lệ.

In [3]:
VALID_SLEEP = {
    'Less than 5 hours',
    '5-6 hours',
    '7-8 hours',
    'More than 8 hours',
}

VALID_DIET = {'Healthy', 'Moderate', 'Unhealthy'}

# --- Train ---
print('=== Cleaning Train ===')
train = clean_categorical_column(train, 'Sleep Duration', VALID_SLEEP)
train = clean_categorical_column(train, 'Dietary Habits', VALID_DIET)

# Extract fill values FROM TRAIN to avoid leakage when cleaning test
# (mode must be computed on train only, not on test)
train_sleep_mode = train['Sleep Duration'].mode()[0]
train_diet_mode  = train['Dietary Habits'].mode()[0]
print(f'\nFill values derived from train: sleep_mode={train_sleep_mode!r}, diet_mode={train_diet_mode!r}')

# --- Test ---
print('\n=== Cleaning Test (using train fill values) ===')
test = clean_categorical_column(test, 'Sleep Duration', VALID_SLEEP, fill_value=train_sleep_mode)
test = clean_categorical_column(test, 'Dietary Habits', VALID_DIET,  fill_value=train_diet_mode)

# Sanity check
print('\nSleep Duration unique (train):', sorted(train['Sleep Duration'].unique()))
print('Dietary Habits unique (train):', sorted(train['Dietary Habits'].unique()))


=== Cleaning Train ===
  [Sleep Duration] dirty values replaced: 79 | pre-existing NaN: 0 | filled with 'Less than 5 hours' | remaining NaN: 0
  [Dietary Habits] dirty values replaced: 23 | pre-existing NaN: 4 | filled with 'Moderate' | remaining NaN: 0

Fill values derived from train: sleep_mode='Less than 5 hours', diet_mode='Moderate'

=== Cleaning Test (using train fill values) ===
  [Sleep Duration] dirty values replaced: 54 | pre-existing NaN: 0 | filled with 'Less than 5 hours' | remaining NaN: 0
  [Dietary Habits] dirty values replaced: 25 | pre-existing NaN: 5 | filled with 'Moderate' | remaining NaN: 0

Sleep Duration unique (train): ['5-6 hours', '7-8 hours', 'Less than 5 hours', 'More than 8 hours']
Dietary Habits unique (train): ['Healthy', 'Moderate', 'Unhealthy']


**Nhận xét**: Sau khi làm sạch, `Sleep Duration` và `Dietary Habits` chỉ còn đúng 4 và 3 category hợp lệ. Số mẫu bị ảnh hưởng rất nhỏ (~79 và ~27 mẫu trên tổng 140,700) — không làm thay đổi phân phối chung nhưng ngăn các category giả làm nhiễu quá trình encoding.

---
## 2.5 — Xử Lý Missing Values Thưa

Các cột này chỉ có rất ít giá trị missing (không phải do cấu trúc conditional):

| Cột | Số NaN | Chiến lược | Lý do |
|-----|--------|------------|-------|
| `Financial Stress` | 4 | Median | Thang đo 1–5 liên tục; median robust hơn mean khi có outliers |
| `Dietary Habits` | 4 (sau clean step) | Mode | Categorical — mode là lựa chọn tự nhiên |
| `Degree` | 2 | Constant `'Unknown'` | Sẽ được Target Encoding ở NB03; `'Unknown'` nhận global mean encoding |

In [4]:
print('Missing values before fill:')
cols_check = ['Financial Stress', 'Dietary Habits', 'Degree']
print(train[cols_check].isnull().sum(), '\n')

# Derive fill statistics from TRAIN only to prevent test leakage
train_fs_median = train['Financial Stress'].median()
train_diet_mode = train['Dietary Habits'].mode()[0]
print(f'Fill stats from train: Financial Stress median={train_fs_median}, Dietary Habits mode={train_diet_mode!r}\n')

print('=== Filling Train ===')
train = fill_sparse_missing(train, 'Financial Stress', strategy='median')
train = fill_sparse_missing(train, 'Dietary Habits',   strategy='mode')
train = fill_sparse_missing(train, 'Degree',           strategy='constant', constant='Unknown')

print('\n=== Filling Test (using train-derived fill values) ===')
test = fill_sparse_missing(test, 'Financial Stress', strategy='constant', constant=train_fs_median)
test = fill_sparse_missing(test, 'Dietary Habits',   strategy='constant', constant=train_diet_mode)
test = fill_sparse_missing(test, 'Degree',           strategy='constant', constant='Unknown')

print('\nMissing values after fill (train):')
print(train[cols_check].isnull().sum())


Missing values before fill:
Financial Stress    4
Dietary Habits      0
Degree              2
dtype: int64 

Fill stats from train: Financial Stress median=3.0, Dietary Habits mode='Moderate'

=== Filling Train ===
  [Financial Stress] Filled 4 NaN with median='3.0' | remaining NaN: 0
  [Dietary Habits] No missing values — skipping.
  [Degree] Filled 2 NaN with constant='Unknown' | remaining NaN: 0

=== Filling Test (using train-derived fill values) ===
  [Financial Stress] No missing values — skipping.
  [Dietary Habits] No missing values — skipping.
  [Degree] Filled 2 NaN with constant='Unknown' | remaining NaN: 0

Missing values after fill (train):
Financial Stress    0
Dietary Habits      0
Degree              0
dtype: int64


**Nhận xét**: Số lượng missing thưa quá nhỏ (< 0.003%) để ảnh hưởng đến phân phối. Việc fill bằng median/mode là an toàn và không gây ra bias đáng kể.

---
## 2.6 — Mã Hóa Binary Features

Các cột sau chỉ có 2 giá trị — áp dụng Label Encoding (0/1) với mapping tường minh:

| Cột | 0 | 1 | Ghi chú |
|-----|---|---|---------|
| `Gender` | Female | Male | Depression rate gần như bằng nhau (17.8% vs 18.5%) |
| `Have you ever had suicidal thoughts ?` | No | Yes | **Predictor mạnh nhất** (31.8% vs 4.9%) |
| `Family History of Mental Illness` | No | Yes | Yếu đơn lẻ nhưng có interaction với suicidal thoughts |
| `Working Professional or Student` | Student | Working Professional | **Cần encode trước** — dùng trong merge step tiếp theo |

> **Tại sao encode `Working Professional or Student` ngay bây giờ?**  
> Bước 2.7 (merge conditional columns) dùng cột này như một boolean mask. Encoding trước giúp điều kiện hóa rõ ràng và nhất quán.

In [5]:
BINARY_MAPPINGS = {
    'Gender': {
        'Female': 0,
        'Male': 1,
    },
    'Have you ever had suicidal thoughts ?': {
        'No': 0,
        'Yes': 1,
    },
    'Family History of Mental Illness': {
        'No': 0,
        'Yes': 1,
    },
    'Working Professional or Student': {
        'Student': 0,
        'Working Professional': 1,
    },
}

print('=== Encoding Train ===')
train = encode_binary_features(train, BINARY_MAPPINGS)

print('\n=== Encoding Test ===')
test = encode_binary_features(test, BINARY_MAPPINGS)

# Verify
encoded_cols = list(BINARY_MAPPINGS.keys())
print('\nSample after encoding:')
train[encoded_cols].head(3)

=== Encoding Train ===
  [Gender] Encoded: {'Female': 0, 'Male': 1}
  [Have you ever had suicidal thoughts ?] Encoded: {'No': 0, 'Yes': 1}
  [Family History of Mental Illness] Encoded: {'No': 0, 'Yes': 1}
  [Working Professional or Student] Encoded: {'Student': 0, 'Working Professional': 1}

=== Encoding Test ===
  [Gender] Encoded: {'Female': 0, 'Male': 1}
  [Have you ever had suicidal thoughts ?] Encoded: {'No': 0, 'Yes': 1}
  [Family History of Mental Illness] Encoded: {'No': 0, 'Yes': 1}
  [Working Professional or Student] Encoded: {'Student': 0, 'Working Professional': 1}

Sample after encoding:


,Gender,Have you ever had suicidal thoughts ?,Family History of Mental Illness,Working Professional or Student
0,0,0,0,1
1,1,1,0,1
2,1,1,0,0


**Nhận xét**: Label Encoding phù hợp cho tất cả 4 cột vì:
- Mỗi cột chỉ có đúng 2 giá trị (binary) — không mất thông tin khi encode.
- One-Hot Encoding sẽ tạo ra cột thừa hoàn toàn tương quan ngược chiều với cột còn lại (multicollinearity hoàn hảo).
- Các tree-based models (RF, XGBoost, LightGBM) xử lý binary integers hiệu quả.

---
## 2.7 — Merge Conditional Columns

Đây là bước **quan trọng nhất và phức tạp nhất** của preprocessing.

### Vấn đề
Dataset có 5 cột chỉ có ý nghĩa với một nhóm đối tượng:
- **Chỉ dành cho Student**: `Academic Pressure`, `CGPA`, `Study Satisfaction`
- **Chỉ dành cho Working Professional**: `Work Pressure`, `Job Satisfaction`

Filling bằng 0 hoặc median toàn cục sẽ tạo ra **dữ liệu giả** (fabricated values) — ví dụ: một kỹ sư 40 tuổi sẽ có `Academic Pressure = 3.14` dù họ không đang học.

### Giải pháp: Merge Strategy

$$\text{Pressure}_i = \begin{cases} \text{Academic Pressure}_i & \text{nếu } \text{role}_i = \text{Student} \\ \text{Work Pressure}_i & \text{nếu } \text{role}_i = \text{Working Professional} \end{cases}$$

$$\text{Satisfaction}_i = \begin{cases} \text{Study Satisfaction}_i & \text{nếu } \text{role}_i = \text{Student} \\ \text{Job Satisfaction}_i & \text{nếu } \text{role}_i = \text{Working Professional} \end{cases}$$

Với `CGPA`: tạo flag `has_cgpa` để model biết đây là thông tin của Student, rồi fill `0.0` cho Professional.

**Kết quả**: 4 cột cũ → 2 cột mới (`Pressure`, `Satisfaction`) + 1 flag (`has_cgpa`) — semantically coherent, zero NaN.

In [6]:
print('Columns before merge:', [c for c in train.columns if 'Pressure' in c or 'Satisfaction' in c or c == 'CGPA'])

print('\n=== Merging Train ===')
train = merge_conditional_columns(train, role_col='Working Professional or Student')

print('\n=== Merging Test ===')
test = merge_conditional_columns(test, role_col='Working Professional or Student')

print('\nColumns after merge:', [c for c in train.columns if 'Pressure' in c or 'Satisfaction' in c or c == 'CGPA' or c == 'has_cgpa'])
print('\nPressure  — describe:')
print(train['Pressure'].describe().round(2))
print('\nSatisfaction — describe:')
print(train['Satisfaction'].describe().round(2))

Columns before merge: ['Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction']

=== Merging Train ===
  [WARN] Pressure had 29 NaN; filled with median=3.00.
  [WARN] Satisfaction had 27 NaN; filled with median=3.00.
  [WARN] 9 student(s) had NaN CGPA; filled with student median=7.77.
  Filled 112793 professional CGPA NaN with sentinel 0.0.
  Merged → 'Pressure' (NaN: 0), 'Satisfaction' (NaN: 0)
  Created → 'has_cgpa' (students: 27901, professionals: 112799)
  Dropped original conditional columns: ['Academic Pressure', 'Work Pressure', 'Study Satisfaction', 'Job Satisfaction']

=== Merging Test ===
  [WARN] Pressure had 17 NaN; filled with median=3.00.
  [WARN] Satisfaction had 17 NaN; filled with median=3.00.
  [WARN] 9 student(s) had NaN CGPA; filled with student median=7.80.
  Filled 75025 professional CGPA NaN with sentinel 0.0.
  Merged → 'Pressure' (NaN: 0), 'Satisfaction' (NaN: 0)
  Created → 'has_cgpa' (students: 18772, professionals: 75028)
  Dro

**Nhận xét**: 
- `Pressure` bao phủ toàn bộ thang đo 1–5 và phản ánh áp lực thực sự của từng người (học tập hoặc công việc).
- `Satisfaction` tương tự — đại diện cho mức độ hài lòng phù hợp với bối cảnh của họ.
- Việc unify 2 cặp conditional thành 2 feature liên tục không chỉ giải quyết missing mà còn **tăng signal** vì áp lực (dù là học hay làm) và sự hài lòng (dù là học hay làm) đều liên quan đến depression theo cùng một chiều.
- `has_cgpa = 1` đánh dấu 27,901 sinh viên — cho model biết `CGPA = 0` ở professional rows là giá trị sentinel, không phải điểm thật.

---
## 2.8 — Mã Hóa Ordinal Features

Hai cột có thứ tự tự nhiên được xác nhận bằng bivariate analysis:

### Sleep Duration
Ngủ nhiều hơn → tỷ lệ depression thấp hơn (gradient rõ ràng):

| Giá trị | Mã hóa | Depression rate |
|---------|--------|----------------|
| Less than 5 hours | 0 | 23.5% |
| 5-6 hours | 1 | 16.6% |
| 7-8 hours | 2 | 17.8% |
| More than 8 hours | 3 | 13.9% |

### Dietary Habits
Chế độ ăn lành mạnh hơn → tỷ lệ depression thấp hơn:

| Giá trị | Mã hóa | Depression rate |
|---------|--------|----------------|
| Unhealthy | 0 | 26.1% |
| Moderate | 1 | 16.6% |
| Healthy | 2 | 11.8% |

> **Tại sao không dùng One-Hot Encoding?**  
> Cả hai cột đều có thứ tự tự nhiên được xác nhận empirically. Ordinal encoding bảo toàn thông tin thứ tự đó và tránh tạo các cột sparse thừa.

In [7]:
ORDINAL_MAPPINGS = {
    'Sleep Duration': [
        'Less than 5 hours',   # 0 — highest risk
        '5-6 hours',           # 1
        '7-8 hours',           # 2
        'More than 8 hours',   # 3 — lowest risk
    ],
    'Dietary Habits': [
        'Unhealthy',   # 0 — highest risk
        'Moderate',    # 1
        'Healthy',     # 2 — lowest risk
    ],
}

print('=== Encoding Train ===')
train = encode_ordinal_features(train, ORDINAL_MAPPINGS)

print('\n=== Encoding Test ===')
test = encode_ordinal_features(test, ORDINAL_MAPPINGS)

print('\nSleep Duration value counts (train):')
print(train['Sleep Duration'].value_counts().sort_index())
print('\nDietary Habits value counts (train):')
print(train['Dietary Habits'].value_counts().sort_index())

=== Encoding Train ===
  [Sleep Duration] Ordinal encoding applied: {'Less than 5 hours': 0, '5-6 hours': 1, '7-8 hours': 2, 'More than 8 hours': 3}
  [Dietary Habits] Ordinal encoding applied: {'Unhealthy': 0, 'Moderate': 1, 'Healthy': 2}

=== Encoding Test ===
  [Sleep Duration] Ordinal encoding applied: {'Less than 5 hours': 0, '5-6 hours': 1, '7-8 hours': 2, 'More than 8 hours': 3}
  [Dietary Habits] Ordinal encoding applied: {'Unhealthy': 0, 'Moderate': 1, 'Healthy': 2}

Sleep Duration value counts (train):
Sleep Duration
0    38863
1    32142
2    36969
3    32726
Name: count, dtype: Int64

Dietary Habits value counts (train):
Dietary Habits
0    46227
1    49732
2    44741
Name: count, dtype: Int64


**Ghi chú về Scaling (dành cho NB04)**:

Numerical features không được scale ở bước này. Scaling sẽ được áp dụng **bên trong `sklearn.Pipeline`** ở NB04 riêng cho từng model:

$$z = \frac{x - \mu}{\sigma} \quad \text{(StandardScaler — dùng cho Logistic Regression, SVM)}$$

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}} \quad \text{(MinMaxScaler — thay thế khi cần giá trị trong [0, 1])}$$

**Lý do deferred scaling**:
- Tree-based models (Random Forest, XGBoost, LightGBM) **không cần** scaling — mọi split chỉ dựa trên ngưỡng.
- Nếu scale ở đây, ta sẽ fit `StandardScaler` trên toàn bộ train — khi dùng trong CV, scaler sẽ "biết" thông tin từ validation fold → **data leakage**.
- `Pipeline` trong sklearn tự động fit scaler chỉ trên training fold, đảm bảo không leakage.

---
## 2.9 — Validation Checks

Trước khi lưu, chạy bộ kiểm tra tự động để đảm bảo data contract:

In [8]:
# Define the expected final schema for train
EXPECTED_TRAIN_COLS = [
    'Gender',
    'Age',
    'City',
    'Working Professional or Student',
    'Profession',
    'Sleep Duration',
    'Dietary Habits',
    'Degree',
    'Have you ever had suicidal thoughts ?',
    'Family History of Mental Illness',
    'Work/Study Hours',
    'Financial Stress',
    'Pressure',
    'Satisfaction',
    'CGPA',
    'has_cgpa',
    'Depression',
]

EXPECTED_TEST_COLS = [c for c in EXPECTED_TRAIN_COLS if c != 'Depression']

# High-cardinality columns kept as strings for Target Encoding in NB03.
# Also allowed to have NaN (e.g., Profession is NaN for students in some rows).
HIGH_CARDINALITY_COLS = ['City', 'Profession', 'Degree']
NULLABLE_COLS = ['City', 'Profession', 'Degree']

print('Validating TRAIN...')
validate_preprocessed_data(
    train,
    expected_columns=EXPECTED_TRAIN_COLS,
    high_cardinality_cols=HIGH_CARDINALITY_COLS,
    nullable_cols=NULLABLE_COLS,
    is_train=True,
)

print('Validating TEST...')
validate_preprocessed_data(
    test,
    expected_columns=EXPECTED_TEST_COLS,
    high_cardinality_cols=HIGH_CARDINALITY_COLS,
    nullable_cols=NULLABLE_COLS,
    is_train=False,
)

Validating TRAIN...

=== Validation Report ===
  PASS  Columns match expected schema (17 cols).
  PASS  Zero NaN in non-nullable columns (allowed NaN in: ['City', 'Profession', 'Degree']).
  PASS  'City' is string dtype (kept for Target Encoding).
  PASS  'Profession' is string dtype (kept for Target Encoding).
  PASS  'Degree' is string dtype (kept for Target Encoding).
  PASS  'Gender' contains only {0, 1}.
  PASS  'Have you ever had suicidal thoughts ?' contains only {0, 1}.
  PASS  'Family History of Mental Illness' contains only {0, 1}.
  PASS  'Working Professional or Student' contains only {0, 1}.
  PASS  'has_cgpa' contains only {0, 1}.
  PASS  Sleep Duration values in {0, 1, 2, 3}.
  PASS  Dietary Habits values in {0, 1, 2}.
  PASS  'Pressure' values in [1.0, 5.0] (min=1.0, max=5.0).
  PASS  'Satisfaction' values in [1.0, 5.0] (min=1.0, max=5.0).
  PASS  'id' correctly absent.
  PASS  'Name' correctly absent.
  PASS  'Academic Pressure' correctly absent.
  PASS  'Work Pressure

---
## 2.10 — Tổng Quan Dataset Sau Preprocessing

In [9]:
print('=== Train — Final State ===')
print(f'Shape: {train.shape}')
print()
print(train.dtypes.to_string())
print()
print('NaN per column:')
print(train.isnull().sum().to_string())
print()
train.describe(include='all').round(2)

=== Train — Final State ===
Shape: (140700, 17)

Gender                                     Int64
Age                                      float64
City                                         str
Working Professional or Student            Int64
Profession                                   str
CGPA                                     float64
Sleep Duration                             Int64
Dietary Habits                             Int64
Degree                                       str
Have you ever had suicidal thoughts ?      Int64
Work/Study Hours                         float64
Financial Stress                         float64
Family History of Mental Illness           Int64
Depression                                 int64
Pressure                                 float64
Satisfaction                             float64
has_cgpa                                   int64

NaN per column:
Gender                                       0
Age                                          0
City   

,Gender,Age,City,Working Professional or Student,Profession,CGPA,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression,Pressure,Satisfaction,has_cgpa
count,140700.0,140700.00,140700,140700.0,104070,140700.00,140700.0,140700.0,140700,140700.0,140700.00,140700.00,140700.0,140700.00,140700.00,140700.00,140700.0
unique,<NA>,NaN,98,<NA>,64,NaN,<NA>,<NA>,116,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN
top,<NA>,NaN,Kalyan,<NA>,Teacher,NaN,<NA>,<NA>,Class 12,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN
freq,<NA>,NaN,6591,<NA>,24906,NaN,<NA>,<NA>,14729,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN
mean,0.55,40.39,NaN,0.8,NaN,1.52,1.45,0.99,NaN,0.49,6.25,2.99,0.5,0.18,3.03,2.97,0.2
std,0.5,12.38,NaN,0.4,NaN,3.12,1.12,0.8,NaN,0.5,3.85,1.41,0.5,0.39,1.40,1.41,0.4
min,0.0,18.00,NaN,0.0,NaN,0.00,0.0,0.0,NaN,0.0,0.00,1.00,0.0,0.00,1.00,1.00,0.0
25%,0.0,29.00,NaN,1.0,NaN,0.00,0.0,0.0,NaN,0.0,3.00,2.00,0.0,0.00,2.00,2.00,0.0
50%,1.0,42.00,NaN,1.0,NaN,0.00,1.0,1.0,NaN,0.0,6.00,3.00,0.0,0.00,3.00,3.00,0.0
75%,1.0,51.00,NaN,1.0,NaN,0.00,2.0,2.0,NaN,1.0,10.00,4.00,1.0,0.00,4.00,4.00,0.0


In [10]:
print('=== Column Summary ===')
summary_rows = []
for col in train.columns:
    summary_rows.append({
        'Column': col,
        'Dtype': str(train[col].dtype),
        'Unique': train[col].nunique(),
        'NaN': train[col].isnull().sum(),
        'Sample values': str(sorted(train[col].dropna().unique()[:5].tolist())),
    })
pd.DataFrame(summary_rows).set_index('Column')

=== Column Summary ===


,Dtype,Unique,NaN,Sample values
Column,,,,
Gender,Int64,2,0,"[0, 1]"
Age,float64,43,0,"[22.0, 26.0, 30.0, 33.0, 49.0]"
City,str,98,0,"['Kanpur', 'Ludhiana', 'Mumbai', 'Varanasi', '..."
Working Professional or Student,Int64,2,0,"[0, 1]"
Profession,str,64,36630,"['Business Analyst', 'Chef', 'Chemist', 'Finan..."
CGPA,float64,332,0,"[0.0, 5.59, 5.9, 7.03, 8.97]"
Sleep Duration,Int64,4,0,"[0, 1, 2, 3]"
Dietary Habits,Int64,3,0,"[0, 1, 2]"
Degree,str,116,0,"['B.Pharm', 'BBA', 'BHM', 'LLB', 'MCA']"


**Nhận xét tổng quan**:
- Dataset giữ nguyên **140,700 rows** — không có hàng nào bị drop trong quá trình preprocessing.
- Từ **20 cột** ban đầu → **17 cột** sau preprocessing:
  - Xóa: `id`, `Name` (2 cột)
  - Xóa: `Academic Pressure`, `Work Pressure`, `Study Satisfaction`, `Job Satisfaction` (4 cột conditional)
  - Thêm: `Pressure`, `Satisfaction`, `has_cgpa` (3 cột mới)
- Tất cả encoded columns là integer, numerical columns là float — kiểu dữ liệu nhất quán.
- 3 cột high-cardinality (`City`, `Profession`, `Degree`) giữ nguyên string để Target Encoding ở NB03.
- **Zero NaN** trên toàn bộ dataset — sẵn sàng cho Feature Engineering.

---
## 2.11 — Lưu Processed Dataset

In [11]:
save_processed_data(
    train_df=train,
    test_df=test,
    output_dir='../data/processed',
    train_ids=train_ids,
    test_ids=test_ids,
    interim_dir='../data/interim',
)

print('\nPreprocessing complete. Outputs saved to data/processed/')
print('Next step: 03_feature_engineering.ipynb')

  Saved: ../data/processed\train_preprocessed.parquet  (951.7 KB, shape=(140700, 17))
  Saved: ../data/processed\test_preprocessed.parquet  (615.4 KB, shape=(93800, 16))
  Saved: ../data/processed\train_preprocessed.csv  (8898.5 KB, shape=(140700, 17))
  Saved: ../data/processed\test_preprocessed.csv  (5747.2 KB, shape=(93800, 16))
  Saved: ../data/interim\train_ids.csv  (140700 IDs)
  Saved: ../data/interim\test_ids.csv  (93800 IDs)

Preprocessing complete. Outputs saved to data/processed/
Next step: 03_feature_engineering.ipynb


---
## 2.12 — Kết Luận

### Tóm tắt các bước đã thực hiện

| Bước | Thao tác | Kết quả |
|------|----------|----------|
| 2.3 | Drop `id`, `Name` | 20 → 18 cột; IDs lưu riêng |
| 2.4 | Clean `Sleep Duration` | 36 → 4 unique values |
| 2.4 | Clean `Dietary Habits` | 23 → 3 unique values |
| 2.5 | Fill sparse missing | Zero NaN ở `Financial Stress`, `Dietary Habits`, `Degree` |
| 2.6 | Binary encoding | 4 cột → 0/1 integer |
| 2.7 | Merge conditional columns | 4 cột → `Pressure` + `Satisfaction` + `has_cgpa` |
| 2.8 | Ordinal encoding | `Sleep Duration` → 0-3; `Dietary Habits` → 0-2 |

### Những gì KHÔNG làm ở bước này
- **Target Encoding** cho `City`, `Profession`, `Degree` → NB03 (cần cross-validation để tránh leakage)
- **Feature Engineering** (interaction features, binning) → NB03
- **Scaling** → NB04 (inject qua `Pipeline` để tránh data leakage trong CV)
- **Imbalance handling** → NB04 (dùng `class_weight='balanced'`)

### Output cho NB03
- `data/processed/train_preprocessed.parquet` — 140,700 × 17
- `data/processed/test_preprocessed.parquet` — N × 16 (không có `Depression`)
- `data/interim/train_ids.csv`, `data/interim/test_ids.csv` — cho NB06 submission